In [64]:
import pdfplumber
import re
import json
import numpy as np
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain_huggingface import HuggingFaceEmbeddings

#### **Extract text from the PDF**

In [65]:
def extract_text_from_pdf(file_path):
    with pdfplumber.open(file_path) as pdf:
        data = pdf.pages
        text = " "
        for page in data:
        # print(page.extract_text())
        # break

            text += page.extract_text() +"\n"
    return text

#### **Clean Text**

In [66]:
def clean_text(text):

    return text.replace("(cid:127)","•")

#### **split the product chunks**

In [67]:
def split_products(text):

    products = re.split("\n(?=[^ \n].+?\nBrand:)",text)

    return [p.strip() for p in products]

#### **Structure chunks into json**

In [68]:
def structure_products(products):

    structured_data = []

    for item in products:

        lines = [l.strip() for l in item.split("\n") if l.strip()]
        title = lines[0]
        brand = next((l.split(":")[1].strip() for l in lines if l.startswith("Brand:")), "")
        price = next((l.split(":")[1].strip() for l in lines if l.startswith("Price:")), "")
        
        structured_data.append({
            "title": title,
            "brand": brand,
            "price": price,
            "content": " ".join(lines)   # full block of text for embeddings
        })
    return structured_data


In [69]:
with open("products.json", "r") as f:
    products = json.load(f)

In [70]:
def generate_embeddings(data, model="sentence-transformers/all-MiniLM-L6-v2"):
    embedder = HuggingFaceEmbeddings(
        model_name=model,
        model_kwargs={"device": "cpu"})

    # Convert JSON into LangChain Document objects
    docs = []
    for item in data:
        docs.append(
            Document(
                page_content=item["content"],  # the main text for embeddings
                metadata={
                    "title": item["title"],
                    "brand": item["brand"],
                    "price": item["price"]
                }
            )
        )

    # Build FAISS index
    vectorstore = FAISS.from_documents(docs, embedder)

    # Save locally
    vectorstore.save_local("faiss_store", index_name="products_index")

    print("✅ FAISS vector store built and saved at faiss_store/products_index")

    return vectorstore

In [71]:
pdf_path = "/home/farheens/Desktop/Order_Management_System/app/documents/product-inquriy.pdf"

In [72]:
# Step 1: Extract & clean text
raw_text = extract_text_from_pdf(pdf_path)

In [73]:
raw_text

' Smart LED TV 55"\nBrand: Samsung\nPrice: 599.99 USD\nAbout this item:\n(cid:127) 4K Ultra HD resolution for crystal-clear viewing\n(cid:127) Smart TV with built-in Wi-Fi and streaming apps\n(cid:127) HDR support for vivid colors and contrast\n(cid:127) Multiple HDMI and USB ports for connectivity\nTechnical Details:\nSpecification Details\nDisplay Size 55 inches\nResolution 3840 x 2160 (4K UHD)\nConnectivity Wi-Fi, HDMI, USB\nAudio Dolby Digital Surround\nDimensions 48.7 x 28.1 x 2.3 inches\nWeight 15.5 kg\nWarranty 1 Year Manufacturer Warranty\nWhat\'s in the Box Smart LED TV 55", User Manual, Warranty Card\nWireless Bluetooth Headphones\nBrand: Sony\nPrice: 199.99 USD\nAbout this item:\n(cid:127) Noise-cancelling over-ear design\n(cid:127) Up to 30 hours of battery life\n(cid:127) Bluetooth 5.0 wireless connectivity\n(cid:127) Comfort-fit cushioned earcups\nTechnical Details:\nSpecification Details\nType Over-Ear\nConnectivity Bluetooth 5.0\nBattery Life 30 hours\nCharging Time 2 h

In [74]:

clean = clean_text(raw_text)

In [75]:
clean

' Smart LED TV 55"\nBrand: Samsung\nPrice: 599.99 USD\nAbout this item:\n• 4K Ultra HD resolution for crystal-clear viewing\n• Smart TV with built-in Wi-Fi and streaming apps\n• HDR support for vivid colors and contrast\n• Multiple HDMI and USB ports for connectivity\nTechnical Details:\nSpecification Details\nDisplay Size 55 inches\nResolution 3840 x 2160 (4K UHD)\nConnectivity Wi-Fi, HDMI, USB\nAudio Dolby Digital Surround\nDimensions 48.7 x 28.1 x 2.3 inches\nWeight 15.5 kg\nWarranty 1 Year Manufacturer Warranty\nWhat\'s in the Box Smart LED TV 55", User Manual, Warranty Card\nWireless Bluetooth Headphones\nBrand: Sony\nPrice: 199.99 USD\nAbout this item:\n• Noise-cancelling over-ear design\n• Up to 30 hours of battery life\n• Bluetooth 5.0 wireless connectivity\n• Comfort-fit cushioned earcups\nTechnical Details:\nSpecification Details\nType Over-Ear\nConnectivity Bluetooth 5.0\nBattery Life 30 hours\nCharging Time 2 hours\nWeight 250 g\nWarranty 1 Year Manufacturer Warranty\nWhat\

In [76]:
# Step 2: Split into product blocks
product_chunks = split_products(clean)
print(f"Extracted {len(product_chunks)} products")


Extracted 10 products


In [77]:
product_chunks[0]

'Smart LED TV 55"\nBrand: Samsung\nPrice: 599.99 USD\nAbout this item:\n• 4K Ultra HD resolution for crystal-clear viewing\n• Smart TV with built-in Wi-Fi and streaming apps\n• HDR support for vivid colors and contrast\n• Multiple HDMI and USB ports for connectivity\nTechnical Details:\nSpecification Details\nDisplay Size 55 inches\nResolution 3840 x 2160 (4K UHD)\nConnectivity Wi-Fi, HDMI, USB\nAudio Dolby Digital Surround\nDimensions 48.7 x 28.1 x 2.3 inches\nWeight 15.5 kg\nWarranty 1 Year Manufacturer Warranty\nWhat\'s in the Box Smart LED TV 55", User Manual, Warranty Card'

In [59]:
# Step 3: Structure into JSON
structured_data = structure_products(product_chunks)
structured_data[0]

{'title': 'Smart LED TV 55"',
 'brand': 'Samsung',
 'price': '599.99 USD',
 'content': 'Smart LED TV 55" Brand: Samsung Price: 599.99 USD About this item: • 4K Ultra HD resolution for crystal-clear viewing • Smart TV with built-in Wi-Fi and streaming apps • HDR support for vivid colors and contrast • Multiple HDMI and USB ports for connectivity Technical Details: Specification Details Display Size 55 inches Resolution 3840 x 2160 (4K UHD) Connectivity Wi-Fi, HDMI, USB Audio Dolby Digital Surround Dimensions 48.7 x 28.1 x 2.3 inches Weight 15.5 kg Warranty 1 Year Manufacturer Warranty What\'s in the Box Smart LED TV 55", User Manual, Warranty Card'}

In [78]:
# Step 4: Save JSON for inspection
with open("products.json", "w") as f:
    json.dump(structured_data, f, indent=2)

In [79]:
vs = generate_embeddings(products)
vs

✅ FAISS vector store built and saved at faiss_store/products_index


In [80]:
query = "Which products support Wi-Fi?"
results = vs.similarity_search(query, k=3)

print("\n🔎 Query:", query)
for r in results:
    print(f"- {r.metadata['title']} | Brand: {r.metadata['brand']} | Price: {r.metadata['price']}")


🔎 Query: Which products support Wi-Fi?
- Wireless Bluetooth Headphones | Brand: Sony | Price: 199.99 USD
- Smartphone X15 | Brand: Apple | Price: 999.0 USD
- Air Purifier Pro | Brand: Dyson | Price: 499.5 USD
